# 4. nn.Module 构建神经网络

PyTorch 中所有神经网络都通过 nn.Module 来构建。

### 核心思想

定义一个神经网络只需要做两件事：
1. __init__() 中定义网络有哪些层（比如全连接层、卷积层）
2. forward() 中定义数据依次经过哪些层（即前向传播的流程）

PyTorch 会自动帮你处理反向传播（Autograd），你不需要手动写。

In [2]:
import torch
import torch.nn as nn

## 4.1 定义一个简单的神经网络

下面定义一个两层的全连接网络：
- 输入 784 维（28x28 的图片展平，比如 MNIST 手写数字）
- 隐藏层 128 维 + ReLU 激活
- 输出 10 维（0-9 共 10 个数字的分类）

In [3]:
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()           # 调用父类的初始化（必须写）
        self.fc1 = nn.Linear(784, 128)  # 第一个全连接层：784 → 128
        self.relu = nn.ReLU()           # 激活函数
        self.fc2 = nn.Linear(128, 10)   # 第二个全连接层：128 → 10

    def forward(self, x):
        x = self.fc1(x)      # 过第一层
        x = self.relu(x)     # 激活
        x = self.fc2(x)      # 过第二层
        return x

# 创建网络实例
model = SimpleNet()
print(model)

SimpleNet(
  (fc1): Linear(in_features=784, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)


In [4]:
# 测试前向传播
# 模拟一个 batch=2 的输入（2张图片，每张784维）
dummy_input = torch.randn(2, 784)
output = model(dummy_input)
print(f"输入形状: {dummy_input.shape}")
print(f"输出形状: {output.shape}")  # (2, 10) — 2张图片，各10个类别的分数
print(f"输出:\n{output}")

输入形状: torch.Size([2, 784])
输出形状: torch.Size([2, 10])
输出:
tensor([[ 0.1589,  0.4124,  0.3227,  0.4328, -0.2623, -0.0935,  0.2212,  0.0661,
         -0.2722, -0.3177],
        [-0.3317,  0.3901,  0.0429,  0.2315, -0.0637,  0.0916,  0.1351,  0.0269,
          0.2249,  0.0841]], grad_fn=<AddmmBackward0>)


## 4.2 常用层介绍

### nn.Linear 全连接层
最基础的层，做 y = xW + b 的线性变换。输入输出都是一维向量。

### nn.Conv2d 卷积层
用于图像处理，自动提取图像特征（边缘、纹理等）。参数：输入通道数、输出通道数、卷积核大小。

### nn.MaxPool2d 池化层
缩小特征图的尺寸，保留最重要的信息。通常和卷积层配合使用。

### nn.Dropout
训练时随机丢弃一部分神经元，防止过拟合。推理时自动关闭。

### nn.Embedding
把离散的索引（比如词语编号）映射成连续的向量。NLP 和 ViT 中常用。

### nn.BatchNorm1d / BatchNorm2d
对每一批数据做归一化，让训练更稳定、收敛更快。

In [5]:
# 常用层演示

# nn.Linear
fc = nn.Linear(10, 5)
x = torch.randn(3, 10)  # batch=3, 特征=10
print(f"Linear: {x.shape} → {fc(x).shape}")

# nn.Conv2d
conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
x = torch.randn(1, 3, 32, 32)  # batch=1, 3通道, 32x32 图片
print(f"Conv2d: {x.shape} → {conv(x).shape}")

# nn.MaxPool2d
pool = nn.MaxPool2d(kernel_size=2)
x = torch.randn(1, 16, 32, 32)
print(f"MaxPool2d: {x.shape} → {pool(x).shape}")

# nn.Dropout
drop = nn.Dropout(p=0.5)  # 50% 概率丢弃
x = torch.randn(3, 10)
print(f"Dropout: {x.shape} → {drop(x).shape}")

# nn.Embedding
embed = nn.Embedding(num_embeddings=1000, embedding_dim=64)
x = torch.tensor([1, 50, 999])  # 3个词语的索引
print(f"Embedding: {x.shape} → {embed(x).shape}")

Linear: torch.Size([3, 10]) → torch.Size([3, 5])
Conv2d: torch.Size([1, 3, 32, 32]) → torch.Size([1, 16, 32, 32])
MaxPool2d: torch.Size([1, 16, 32, 32]) → torch.Size([1, 16, 16, 16])
Dropout: torch.Size([3, 10]) → torch.Size([3, 10])
Embedding: torch.Size([3]) → torch.Size([3, 64])


## 4.3 损失函数

损失函数衡量模型预测和真实标签之间的差距。训练的目标就是最小化损失。

### 核心概念补充：Softmax 与 均方误差

**1. Softmax (在 CrossEntropyLoss 内部自动计算)**
- **作用**：把模型输出的一堆“原始分数”转换成“概率（百分比）”。
- **原理**：通过取指数($e^x$)把所有分数变成正数并且拉大差距，然后再计算各自占总和的百分比。最终输出的所有类别的概率加起来刚好等于 1（100%）。

**2. MSELoss (均方误差 Mean Squared Error)**
- **误差 (Error)**：预测值 - 真实值。
- **方 (Squared)**：将误差平方。一是为了防止正负误差互相抵消；二是为了“狠狠惩罚”那些错得离谱的预测（错得越多，平方后的惩罚越大）。
- **均 (Mean)**：把这一批数据的平方误差加起来，求个平均数。MSE 的值越接近0，说明模型预测越准。

### 常用损失函数
- nn.CrossEntropyLoss — 多分类任务（内部自带 Softmax，直接传原始分数即可）
- nn.MSELoss — 回归任务，均方误差
- nn.BCEWithLogitsLoss — 二分类任务

In [6]:
# 损失函数演示

# CrossEntropyLoss — 多分类
criterion = nn.CrossEntropyLoss()
output = torch.randn(3, 10)     # 3个样本，各10个类别的分数
target = torch.tensor([0, 5, 9]) # 3个样本的真实标签
loss = criterion(output, target)
print(f"CrossEntropyLoss: {loss.item():.4f}")

# MSELoss — 回归
mse = nn.MSELoss()
pred = torch.tensor([1.0, 2.0, 3.0])
true = torch.tensor([1.5, 2.5, 3.5])
loss = mse(pred, true)
print(f"MSELoss: {loss.item():.4f}")

CrossEntropyLoss: 2.3568
MSELoss: 0.2500


## 4.4 优化器

优化器根据梯度来更新模型参数。

- SGD — 最基础的随机梯度下降
- Adam — 自适应学习率，最常用，大多数情况默认选它
- AdamW — 带权重衰减的 Adam，Transformer 训练标配

用法都是一样的：
1. 创建时传入 model.parameters() 和学习率 lr
2. 训练循环中依次调用 zero_grad() → backward() → step()

In [7]:
# 优化器演示

model = SimpleNet()

# 三种优化器对比
optimizer_sgd = torch.optim.SGD(model.parameters(), lr=0.01)
optimizer_adam = torch.optim.Adam(model.parameters(), lr=1e-3)
optimizer_adamw = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

print(f"SGD 参数组数: {len(optimizer_sgd.param_groups)}")
print(f"Adam 参数组数: {len(optimizer_adam.param_groups)}")
print(f"AdamW 参数组数: {len(optimizer_adamw.param_groups)}")

# 查看模型有多少参数
total_params = sum(p.numel() for p in model.parameters())
print(f"\n模型总参数量: {total_params:,}")

SGD 参数组数: 1
Adam 参数组数: 1
AdamW 参数组数: 1

模型总参数量: 101,770
